# Model Training Demo

This notebook demonstrates the complete training workflow:
1. Loading data
2. Setting up base model and LoRA
3. Training configuration
4. Running training
5. Saving and registering the model

**Note:** For production training, use the training script: `python training/train_sft.py`

## Setup

In [ ]:
import sys
sys.path.append('..')

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

from data.data_loader import DatabricksConnector, DataPreprocessor
from training.lora_config import LoRAConfigBuilder
from models.registry import ModelRegistry

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load and Prepare Data

In [ ]:
# Load data
connector = DatabricksConnector(use_simulation=True)
dataset = connector.load_training_data(limit=100)  # Small for demo

# Preprocess
preprocessor = DataPreprocessor(format="alpaca")
prepared_data = preprocessor.prepare_dataset(dataset, train_split=0.9)

train_dataset = prepared_data['train']
eval_dataset = prepared_data['validation']

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")

## 2. Load Base Model and Tokenizer

In [ ]:
model_name = "distilgpt2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model loaded: {total_params:,} parameters")

## 3. Apply LoRA (Parameter-Efficient Fine-Tuning)

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,                      # Rank
    lora_alpha=16,            # Scaling factor
    target_modules=["c_attn", "c_proj"],  # Which layers to adapt
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(base_model, lora_config)

# Show trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Percentage of base model: {100 * trainable_params / total_params:.2f}%")
print(f"\nThis is {total_params / trainable_params:.0f}x more parameter efficient!")

## 4. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"Tokenized {len(tokenized_train)} training examples")
print(f"Tokenized {len(tokenized_eval)} validation examples")

## 5. Configure Training

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/checkpoints/notebook_demo",
    
    # Training hyperparameters
    num_train_epochs=1,                    # 1 epoch for demo
    per_device_train_batch_size=2,        # Small batch for CPU
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,        # Effective batch size = 4
    
    # Optimization
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    
    # Evaluation
    evaluation_strategy="steps",
    eval_steps=20,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,
    
    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    
    # Misc
    report_to="none",  # No external logging for notebook
    seed=42
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Total steps: ~{len(tokenized_train) // training_args.per_device_train_batch_size}")

## 6. Initialize Trainer

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

print("Trainer initialized and ready!")

## 7. Train Model

**Note:** This will take several minutes on CPU. For demonstration purposes, we're only training for 1 epoch on 100 examples.

In [ ]:
# Uncomment to run training
# This is commented out to avoid long execution in the notebook
# For actual training, use the training script instead

# print("Starting training...")
# train_result = trainer.train()
# print("\nTraining complete!")
# print(f"Final loss: {train_result.training_loss:.4f}")

print("\n⚠️ Training commented out for notebook demo")
print("To train the model, run: python training/train_sft.py")

## 8. Save and Register Model

In [ ]:
# Uncomment after training
# trainer.save_model("../models/checkpoints/notebook_demo_final")
# tokenizer.save_pretrained("../models/checkpoints/notebook_demo_final")

# # Register in model registry
# registry = ModelRegistry()
# version = registry.register_model(
#     model_path="../models/checkpoints/notebook_demo_final",
#     model_name="notebook-demo",
#     metrics={
#         "train_loss": train_result.training_loss,
#         "eval_loss": trainer.evaluate()['eval_loss']
#     },
#     metadata={
#         "base_model": "distilgpt2",
#         "lora_rank": 8,
#         "training_samples": len(train_dataset)
#     }
# )

# print(f"Model saved and registered as: {version}")

print("Model saving steps shown above (commented out for demo)")

## 9. Test Inference (Example)

In [ ]:
# Example of how to use the trained model
# (This would work after training is complete)

def generate_response(model, tokenizer, instruction, max_length=128):
    """Generate response for an instruction"""
    
    # Format in Alpaca style
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Example usage (after training)
# test_instruction = "How do I reset my password?"
# response = generate_response(model, tokenizer, test_instruction)
# print(f"Q: {test_instruction}")
# print(f"A: {response}")

print("Inference function defined. Use after training is complete.")

## Summary

This notebook demonstrated:
- ✅ Data loading and preprocessing
- ✅ LoRA configuration (8 rank, 99%+ parameter reduction)
- ✅ Training setup with HuggingFace Trainer
- ✅ Model registration for versioning

**For Production Training:**
```bash
python training/train_sft.py --config training/config.yaml
```

**Next Steps:**
1. Run full training with more epochs and data
2. Evaluate model performance (see `03_evaluation_analysis.ipynb`)
3. Deploy via API: `python api/main.py`